In [ ]:
import json
from datetime import date
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PASTA_PROJETO = Path("/content/drive/MyDrive/Sistema_Biblioteca_Python")
PASTA_PROJETO.mkdir(parents=True, exist_ok=True)
ARQUIVO_DADOS = PASTA_PROJETO / "livros.json"

livros = []


def validar_livro(livro):
    return (
        isinstance(livro, dict)
        and isinstance(livro.get("título"), str)
        and bool(livro["título"].strip())
        and isinstance(livro.get("autor"), str)
        and bool(livro["autor"].strip())
        and type(livro.get("ano")) is int
        and 1 <= livro["ano"] <= date.today().year
        and livro.get("disponível") in ("Sim", "Não")
    )


def carregar_livros():
    try:
        with ARQUIVO_DADOS.open("r", encoding="utf-8") as arquivo:
            dados = json.load(arquivo)

        if not isinstance(dados, list) or not all(
            validar_livro(livro) for livro in dados
        ):
            raise ValueError("O arquivo contém dados inválidos.")

    except FileNotFoundError:
        livros.clear()
        return True
    except (OSError, ValueError, UnicodeError) as erro:
        print(f"Não foi possível carregar o acervo: {erro}")
        print("Confira o arquivo livros.json antes de continuar.")
        return False

    livros[:] = dados
    return True


def salvar_livros(novo_acervo):
    temporario = ARQUIVO_DADOS.with_suffix(".json.tmp")

    try:
        with temporario.open("w", encoding="utf-8") as arquivo:
            json.dump(novo_acervo, arquivo, ensure_ascii=False, indent=4)
            arquivo.write("\n")

        temporario.replace(ARQUIVO_DADOS)

    except OSError as erro:
        print(f"Não foi possível salvar: {erro}")
        print("A alteração não foi aplicada.")

        try:
            temporario.unlink(missing_ok=True)
        except OSError:
            pass

        return False

    livros[:] = novo_acervo
    return True


def ler_texto(mensagem):
    while True:
        texto = input(mensagem).strip()

        if texto:
            return texto

        print("Este campo não pode ficar vazio.")


def ler_inteiro(mensagem, minimo, maximo):
    while True:
        try:
            numero = int(input(mensagem))

            if minimo <= numero <= maximo:
                return numero

        except ValueError:
            pass

        print(f"Digite um número inteiro entre {minimo} e {maximo}.")


def cadastrar_livro():
    titulo = ler_texto("Título: ")
    autor = ler_texto("Autor: ")
    ano = ler_inteiro("Ano de publicação: ", 1, date.today().year)

    livro = {
        "título": titulo,
        "autor": autor,
        "ano": ano,
        "disponível": "Sim"
    }

    if salvar_livros(livros + [livro]):
        print("Livro cadastrado com sucesso!")


def exibir_livros(registros):
    for indice, livro in registros:
        print(f"\nNúmero: {indice + 1}")
        print(f"Título: {livro['título']}")
        print(f"Autor: {livro['autor']}")
        print(f"Ano: {livro['ano']}")
        print(f"Disponível: {livro['disponível']}")


def listar_livros():
    if not livros:
        print("Nenhum livro cadastrado.")
        return

    print("\n1 - Ordem de cadastro")
    print("2 - Ordem alfabética por título")
    print("3 - Ano de publicação")

    ordem = ler_inteiro("Escolha: ", 1, 3)
    registros = list(enumerate(livros))

    if ordem == 2:
        registros.sort(key=lambda item: item[1]["título"].casefold())
    elif ordem == 3:
        registros.sort(key=lambda item: item[1]["ano"])

    exibir_livros(registros)


def buscar_livros():
    if not livros:
        print("Nenhum livro cadastrado.")
        return

    print("\n1 - Buscar por título")
    print("2 - Buscar por autor")

    filtro = ler_inteiro("Escolha: ", 1, 2)
    campo = "título" if filtro == 1 else "autor"
    termo = ler_texto("Digite o termo da busca: ").casefold()

    encontrados = [
        (indice, livro)
        for indice, livro in enumerate(livros)
        if termo in livro[campo].casefold()
    ]

    if encontrados:
        exibir_livros(encontrados)
    else:
        print("Nenhum livro encontrado.")


def selecionar_livro():
    if not livros:
        print("Nenhum livro cadastrado.")
        return None

    exibir_livros(enumerate(livros))

    numero = ler_inteiro(
        "\nNúmero do livro (0 para cancelar): ", 0, len(livros)
    )

    return numero - 1 if numero else None


def alterar_disponibilidade(novo_status):
    indice = selecionar_livro()

    if indice is None:
        return

    if livros[indice]["disponível"] == novo_status:
        if novo_status == "Não":
            print("O livro já está emprestado.")
        else:
            print("O livro já está disponível.")
        return

    novo_acervo = [livro.copy() for livro in livros]
    novo_acervo[indice]["disponível"] = novo_status

    if salvar_livros(novo_acervo):
        if novo_status == "Não":
            print("Empréstimo realizado com sucesso!")
        else:
            print("Devolução realizada com sucesso!")


def emprestar_livro():
    alterar_disponibilidade("Não")


def devolver_livro():
    alterar_disponibilidade("Sim")


def contar_acervo():
    disponiveis = sum(
        1 for livro in livros if livro["disponível"] == "Sim"
    )

    print(f"\nTotal de livros: {len(livros)}")
    print(f"Disponíveis: {disponiveis}")
    print(f"Emprestados: {len(livros) - disponiveis}")


def remover_livro():
    indice = selecionar_livro()

    if indice is None:
        return

    if livros[indice]["disponível"] == "Não":
        print("Devolva o livro antes de removê-lo.")
        return

    confirmacao = input(
        f"Remover '{livros[indice]['título']}'? Digite S para confirmar: "
    ).strip().casefold()

    if confirmacao != "s":
        print("Remoção cancelada.")
        return

    novo_acervo = livros[:indice] + livros[indice + 1:]

    if salvar_livros(novo_acervo):
        print("Livro removido com sucesso!")


def menu_principal():
    while True:
        print("\n=== SISTEMA DE CADASTRO DE BIBLIOTECA ===")
        print("1 - Cadastrar livro")
        print("2 - Listar livros")
        print("3 - Buscar livro")
        print("4 - Emprestar livro")
        print("5 - Devolver livro")
        print("6 - Contador do acervo")
        print("7 - Remover livro")
        print("0 - Sair")

        opcao = ler_inteiro("Escolha uma opção: ", 0, 7)

        if opcao == 0:
            print("Programa encerrado. As alterações já foram salvas.")
            break
        elif opcao == 1:
            cadastrar_livro()
        elif opcao == 2:
            listar_livros()
        elif opcao == 3:
            buscar_livros()
        elif opcao == 4:
            emprestar_livro()
        elif opcao == 5:
            devolver_livro()
        elif opcao == 6:
            contar_acervo()
        elif opcao == 7:
            remover_livro()


def main():
    try:
        if carregar_livros():
            menu_principal()
    except (KeyboardInterrupt, EOFError):
        print("\nPrograma encerrado. Alterações concluídas já estão salvas.")


main()

Mounted at /content/drive

=== SISTEMA DE CADASTRO DE BIBLIOTECA ===
1 - Cadastrar livro
2 - Listar livros
3 - Buscar livro
4 - Emprestar livro
5 - Devolver livro
6 - Contador do acervo
7 - Remover livro
0 - Sair
Escolha uma opção: 1
Título: A Bela e a Fera
Autor: Gabrielle-Suzanne Barbot
Ano de publicação: 1740
Livro cadastrado com sucesso!

=== SISTEMA DE CADASTRO DE BIBLIOTECA ===
1 - Cadastrar livro
2 - Listar livros
3 - Buscar livro
4 - Emprestar livro
5 - Devolver livro
6 - Contador do acervo
7 - Remover livro
0 - Sair
Escolha uma opção: 1
Título: Dom Casmurro 
Autor: Machado de Assis
Ano de publicação: 1899
Livro cadastrado com sucesso!

=== SISTEMA DE CADASTRO DE BIBLIOTECA ===
1 - Cadastrar livro
2 - Listar livros
3 - Buscar livro
4 - Emprestar livro
5 - Devolver livro
6 - Contador do acervo
7 - Remover livro
0 - Sair
Escolha uma opção: 2

1 - Ordem de cadastro
2 - Ordem alfabética por título
3 - Ano de publicação
Escolha: 2

Número: 6
Título: A Bela e A fera
Autor: xxxx
Ano: 